# Tutorial 04 — Gerenciamento de Recursos com Langfuse

> **Importante para o Hackathon:** todos os custos são rastreados via Langfuse session IDs.
> Sem as credenciais configuradas no `.env`, o uso não será contabilizado pela organização.

Este tutorial ensina como monitorar tokens, custos e latência dos seus agentes usando a Langfuse.

## Célula 1 — Instalação

Dois novos pacotes além dos tutoriais anteriores:
- `langfuse` — plataforma de observabilidade que rastreia tokens, custos e latência
- `ulid-py` — gera IDs únicos ordenáveis para identificar cada sessão

In [1]:
%pip install langchain langchain-openai langfuse python-dotenv ulid-py --quiet

Note: you may need to restart the kernel to use updated packages.


## Célula 2 — Configurar o modelo

Mesma configuração dos tutoriais anteriores. O modelo `gpt-4o-mini` será rastreado pelo Langfuse.

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Carrega todas as variáveis do .env (inclusive as do Langfuse)
load_dotenv()

model_id = "gpt-4o-mini"

model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model=model_id,
    temperature=0.7,
    max_tokens=1000,
)

print(f"Modelo configurado: {model_id}")

Modelo configurado: gpt-4o-mini


## Célula 3 — Inicializar Langfuse e funções helper

Três componentes novos:

1. **`Langfuse()`** — cliente que se conecta ao servidor Langfuse usando as credenciais do `.env`
2. **`@observe()`** — decorador que cria um trace automaticamente a cada chamada da função
3. **`CallbackHandler()`** — se encaixa dentro do `@observe()` e captura tudo que o LangChain faz

**Sobre o `session_id`:** agrupa todas as chamadas de uma execução sob um único ID.
Formato exigido pelo Hackathon: `{TEAM_NAME}-{ULID}`

In [3]:
import ulid
from langfuse import Langfuse, observe, propagate_attributes
from langfuse.langchain import CallbackHandler

# Inicializa o cliente Langfuse com as credenciais do .env
langfuse_client = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST", "https://challenges.reply.com/langfuse"),
)

def generate_session_id():
    """Gera um session ID unico no formato {TEAM_NAME}-{ULID}."""
    return f"{os.getenv('TEAM_NAME', 'tutorial')}-{ulid.new().str}"

def invoke_langchain(model, prompt, langfuse_handler):
    """Chama o modelo com o handler do Langfuse para rastrear tokens e custos."""
    messages = [HumanMessage(content=prompt)]
    response = model.invoke(messages, config={"callbacks": [langfuse_handler]})
    return response.content

@observe()
def run_llm_call(session_id, model, prompt):
    """Executa uma chamada ao LLM e registra tudo no Langfuse."""
    # propagate_attributes e uma funcao importada, nao metodo do cliente
    with propagate_attributes(session_id=session_id):
        langfuse_handler = CallbackHandler()
        response = invoke_langchain(model, prompt, langfuse_handler)
        return response

print("Langfuse inicializado com sucesso!")
print(f"Public key: {str(os.getenv('LANGFUSE_PUBLIC_KEY', 'Nao configurada'))[:20]}...")
print(f"Host: {os.getenv('LANGFUSE_HOST')}")
print(f"Team: {os.getenv('TEAM_NAME')}")

Langfuse inicializado com sucesso!
Public key: pk-lf-6b3da0a8-8cc2-...
Host: https://us.cloud.langfuse.com
Team: Solivus-Hub


## Célula 4 — Uma chamada rastreada

O que acontece por baixo dos panos:
1. `@observe()` cria um trace no Langfuse
2. `update_current_trace(session_id=...)` marca o trace com o ID da sessão
3. `CallbackHandler()` captura tokens, custo e latência da chamada
4. `langfuse_client.flush()` garante que os dados são enviados ao servidor

In [4]:
session_id = generate_session_id()
print(f"Session ID: {session_id}\n")

response = run_llm_call(session_id, model, "Qual é a raiz quadrada de 144?")

print(f"Pergunta: Qual é a raiz quadrada de 144?")
print(f"Resposta: {response}")

# Garante que o trace foi enviado ao servidor Langfuse
langfuse_client.flush()

print(f"\nTrace enviado ao Langfuse!")
print(f"Session ID: {session_id}")

Session ID: Solivus-Hub-01KPB8CZ86Z43MJJTF4DJ0QXHR

Pergunta: Qual é a raiz quadrada de 144?
Resposta: A raiz quadrada de 144 é 12.

Trace enviado ao Langfuse!
Session ID: Solivus-Hub-01KPB8CZ86Z43MJJTF4DJ0QXHR


## Célula 5 — Múltiplas chamadas na mesma sessão

O mesmo `session_id` agrupa todas as chamadas.
A Langfuse soma tokens e custos automaticamente — sem precisar acumular manualmente.

In [5]:
perguntas = [
    "O que é machine learning?",
    "Explique redes neurais brevemente.",
    "Qual a diferença entre IA e ML?"
]

session_id = generate_session_id()
print(f"Session ID: {session_id}")
print(f"Fazendo {len(perguntas)} chamadas rastreadas...\n")

for i, pergunta in enumerate(perguntas, 1):
    response = run_llm_call(session_id, model, pergunta)
    print(f"Chamada {i}: {pergunta}")
    print(f"  Resposta: {response[:80]}...\n")

langfuse_client.flush()

print("=" * 50)
print(f"Todas as {len(perguntas)} chamadas enviadas ao Langfuse!")
print(f"Session ID: {session_id}")

Session ID: Solivus-Hub-01KPB8DA7BPSJE93AFWS277KVJ
Fazendo 3 chamadas rastreadas...

Chamada 1: O que é machine learning?
  Resposta: Machine learning, ou aprendizado de máquina, é uma subárea da inteligência artif...

Chamada 2: Explique redes neurais brevemente.
  Resposta: Redes neurais são modelos computacionais inspirados no funcionamento do cérebro ...

Chamada 3: Qual a diferença entre IA e ML?
  Resposta: A Inteligência Artificial (IA) e o Aprendizado de Máquina (ML) são conceitos rel...

Todas as 3 chamadas enviadas ao Langfuse!
Session ID: Solivus-Hub-01KPB8DA7BPSJE93AFWS277KVJ


## Célula 6 — Visualizar traces e custos

Consulta o Langfuse e exibe um resumo da sessão:
quantidade de chamadas, custo por modelo, tempo total e preview do que foi enviado/recebido.

In [6]:
from datetime import datetime
from collections import defaultdict

def get_trace_info(session_id: str):
    """Busca os traces de uma sessão e agrega estatísticas."""
    traces = []
    page = 1
    while True:
        response = langfuse_client.api.trace.list(session_id=session_id, page=page)
        if not response.data:
            break
        traces.extend(response.data)
        if len(response.data) < 100:
            break
        page += 1

    if not traces:
        return None

    observations = []
    for trace in traces:
        detail = langfuse_client.api.trace.get(trace.id)
        if detail and hasattr(detail, "observations"):
            observations.extend(detail.observations)

    if not observations:
        return None

    sorted_obs = sorted(
        observations,
        key=lambda o: o.start_time if hasattr(o, "start_time") and o.start_time else datetime.min
    )

    counts = defaultdict(int)
    costs = defaultdict(float)
    total_time = 0.0

    for obs in observations:
        if hasattr(obs, "type") and obs.type == "GENERATION":
            mdl = getattr(obs, "model", "unknown") or "unknown"
            counts[mdl] += 1
            if hasattr(obs, "calculated_total_cost") and obs.calculated_total_cost:
                costs[mdl] += obs.calculated_total_cost
            if hasattr(obs, "start_time") and hasattr(obs, "end_time"):
                if obs.start_time and obs.end_time:
                    total_time += (obs.end_time - obs.start_time).total_seconds()

    first_input = ""
    if sorted_obs and hasattr(sorted_obs[0], "input"):
        inp = sorted_obs[0].input
        if inp:
            first_input = str(inp)[:100]

    last_output = ""
    if sorted_obs and hasattr(sorted_obs[-1], "output"):
        out = sorted_obs[-1].output
        if out:
            last_output = str(out)[:100]

    return {
        "counts": dict(counts),
        "costs": dict(costs),
        "time": total_time,
        "input": first_input,
        "output": last_output,
    }

def print_results(info):
    """Exibe as estatísticas agregadas da sessão."""
    if not info:
        print("Nenhum trace encontrado para este session_id.")
        return

    print("\nChamadas por modelo:")
    for mdl, count in info["counts"].items():
        print(f"  {mdl}: {count}")

    print("\nCusto por modelo:")
    total = 0.0
    for mdl, cost in info["costs"].items():
        print(f"  {mdl}: ${cost:.6f}")
        total += cost
    if total > 0:
        print(f"  Total: ${total:.6f}")

    print(f"\nTempo total: {info['time']:.2f}s")

    if info["input"]:
        print(f"\nPrimeiro input:\n  {info['input']}")
    if info["output"]:
        print(f"\nÚltimo output:\n  {info['output']}")
    print()

print("Funções de visualização prontas!")
print("Use: info = get_trace_info(session_id) e depois print_results(info)")

Funções de visualização prontas!
Use: info = get_trace_info(session_id) e depois print_results(info)


## Célula 7 — Consultar a sessão

Substitua o `session_id` abaixo pelo ID gerado nas células anteriores para ver os dados.

In [7]:
# Cole aqui o session_id gerado na Célula 4 ou 5
# Exemplo: session_id_consulta = "meu-time-01JSCXYZ123..."
session_id_consulta = session_id  # usa o último session_id gerado

info = get_trace_info("Solivus-Hub-01KPA3APDA47ADAJ4YVVCGF2XP")
print_results(info)


Chamadas por modelo:
  openai/gpt-4o-mini: 3

Custo por modelo:
  openai/gpt-4o-mini: $0.000599
  Total: $0.000599

Tempo total: 15.24s

Primeiro input:
  {'args': ['Solivus-Hub-01KPA3APDA47ADAJ4YVVCGF2XP', {'name': None, 'disable_streaming': False, 'outp

Último output:
  {'role': 'assistant', 'content': 'Inteligência Artificial (IA) e Aprendizado de Máquina (ML) são con

